In [11]:
import pandas as pd
import torch
import os
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm

# =========================
# CONFIG
# =========================
base_model = "google/gemma-2-9b-it"
lora_path = "gemma_mcq_sft_model"

input_file = "final_combined_dataset.csv"
output_file = "abstention_gemma_xml.csv"

save_every = 50
batch_size = 1

# =========================
# LOAD TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    base_model,
    padding_side="left"
)
tokenizer.pad_token = tokenizer.eos_token

# =========================
# LOAD MODEL
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model = PeftModel.from_pretrained(model, lora_path)
model.eval()

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(input_file)

# =========================
# RESUME LOGIC
# =========================
if os.path.exists(output_file):
    df_existing = pd.read_csv(output_file)
    start_idx = len(df_existing)
    print(f"Resuming from row {start_idx}")
else:
    start_idx = 0

# =========================
# PROMPT
# =========================
def build_prompt(row):
    return f"""
You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT):
<answer>a/b/c/d/idk</answer>
<reasoning>Give 2-3 line explanation</reasoning>

Only output these two tags.
"""

# =========================
# PARSER
# =========================
def parse_output(output):
    final_answer = "idk"
    reasoning = ""

    try:
        ans_match = re.search(
            r"<answer>\s*(.*?)\s*</answer>",
            output,
            re.IGNORECASE
        )

        if ans_match:
            final_answer = ans_match.group(1).strip().lower()

        reason_match = re.search(
            r"<reasoning>\s*(.*?)\s*</reasoning>",
            output,
            re.IGNORECASE | re.DOTALL
        )

        if reason_match:
            reasoning = reason_match.group(1).strip()

    except:
        pass

    if final_answer not in ["a", "b", "c", "d", "idk"]:
        final_answer = "idk"

    if reasoning == "":
        final_answer = "idk"

    return final_answer, reasoning

# =========================
# INFERENCE LOOP
# =========================
results_buffer = []
buffer_start_idx = start_idx

for i in tqdm(range(start_idx, len(df), batch_size), desc="Running Inference"):

    batch = df.iloc[i:i + batch_size]
    prompts = [build_prompt(row) for _, row in batch.iterrows()]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=0.0
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for prompt, full_output in zip(prompts, decoded):
        cleaned = full_output[len(prompt):].strip()

        final_answer, reasoning = parse_output(cleaned)

        results_buffer.append({
            "pred_final_answer": final_answer,
            "pred_reasoning": reasoning
        })

    # =========================
    # SAVE EVERY 50 ROWS
    # =========================
    should_save = (
        len(results_buffer) >= save_every
        or (i + batch_size) >= len(df)
    )

    if should_save:

        end_idx = buffer_start_idx + len(results_buffer)

        df_chunk = df.iloc[buffer_start_idx:end_idx].copy()

        df_chunk["pred_final_answer"] = [
            x["pred_final_answer"] for x in results_buffer
        ]

        df_chunk["pred_reasoning"] = [
            x["pred_reasoning"] for x in results_buffer
        ]

        if os.path.exists(output_file):
            df_chunk.to_csv(
                output_file,
                mode="a",
                header=False,
                index=False
            )
        else:
            df_chunk.to_csv(
                output_file,
                index=False
            )

        print(f"Saved rows {buffer_start_idx} to {end_idx}")

        buffer_start_idx = end_idx
        results_buffer = []

print("Done! Fully robust pipeline completed.")

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Resuming from row 3521


Running Inference: 0it [00:00, ?it/s]

Done! Fully robust pipeline completed.


In [16]:
import pandas as pd
import torch
import os
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm

# =====================================================
# GEMMA SFT INFERENCE (BATCH SIZE = 8)
# Resume from row 3450
# Appends safely to CSV
# =====================================================

# =========================
# CONFIG
# =========================
base_model = "google/gemma-2-9b-it"
lora_path = "gemma_mcq_sft_model"

input_file = "final_combined_dataset.csv"
output_file = "abstention_sft_gemma_xml.csv"

start_idx = 3450          # remaining rows start
batch_size = 8
save_every = 8            # save every batch

torch.cuda.empty_cache()

# =========================
# LOAD TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    base_model,
    padding_side="left"
)

tokenizer.pad_token = tokenizer.eos_token

# =========================
# LOAD MODEL
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load LoRA
model = PeftModel.from_pretrained(model, lora_path)

model.eval()

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(input_file)

print(f"Starting from row {start_idx}")
print(f"Remaining rows: {len(df) - start_idx}")

# =========================
# PROMPT
# =========================
def build_prompt(row):
    return f"""
You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT):
<answer>a/b/c/d/idk</answer>
<reasoning>Give 2-3 line explanation</reasoning>

Only output these two tags.
"""

# =========================
# PARSER
# =========================
def parse_output(output):
    final_answer = "idk"
    reasoning = ""

    try:
        ans_match = re.search(
            r"<answer>\s*(.*?)\s*</answer>",
            output,
            re.IGNORECASE
        )

        if ans_match:
            final_answer = ans_match.group(1).strip().lower()

        reason_match = re.search(
            r"<reasoning>\s*(.*?)\s*</reasoning>",
            output,
            re.IGNORECASE | re.DOTALL
        )

        if reason_match:
            reasoning = reason_match.group(1).strip()

    except:
        pass

    if final_answer not in ["a", "b", "c", "d", "idk"]:
        final_answer = "idk"

    if reasoning == "":
        final_answer = "idk"

    return final_answer, reasoning

# =========================
# INFERENCE LOOP
# =========================
results_buffer = []

for i in tqdm(range(start_idx, len(df), batch_size), desc="Running Inference"):

    batch = df.iloc[i:i + batch_size].copy()

    prompts = [build_prompt(row) for _, row in batch.iterrows()]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=0.0
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # =========================
    # PROCESS EACH ROW
    # =========================
    for row_idx, full_output in enumerate(decoded):

        prompt = prompts[row_idx]

        # remove prompt part
        cleaned = full_output[len(prompt):].strip()

        final_answer, reasoning = parse_output(cleaned)

        row_data = batch.iloc[row_idx].copy()

        row_data["pred_final_answer"] = final_answer
        row_data["pred_reasoning"] = reasoning

        results_buffer.append(row_data)

    # =========================
    # SAVE EACH BATCH
    # =========================
    if len(results_buffer) >= save_every:

        save_df = pd.DataFrame(results_buffer)

        if os.path.exists(output_file):
            save_df.to_csv(
                output_file,
                mode="a",
                header=False,
                index=False
            )
        else:
            save_df.to_csv(
                output_file,
                index=False
            )

        results_buffer = []

# =========================
# FINAL LEFTOVER SAVE
# =========================
if len(results_buffer) > 0:

    save_df = pd.DataFrame(results_buffer)

    if os.path.exists(output_file):
        save_df.to_csv(
            output_file,
            mode="a",
            header=False,
            index=False
        )
    else:
        save_df.to_csv(
            output_file,
            index=False
        )

print("Done! Remaining rows completed.")

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Starting from row 3450
Remaining rows: 71


Running Inference: 100%|██████████| 9/9 [04:59<00:00, 33.33s/it]

Done! Remaining rows completed.
